# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [4]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-4o-mini'
openai = OpenAI()

API key looks good so far


In [5]:
links = fetch_website_links("https://segun-jr-agbetuyi.vercel.app/")
links

['#home',
 '#about',
 '#experience',
 '#projects',
 '#skills',
 '#services',
 'https://github.com/zoro-chi',
 'https://www.linkedin.com/in/segun-jr-agbetuyi/',
 'https://segun-jr-agbetuyi.vercel.app/',
 'https://admissions.au.edu/',
 'https://www.bedfordschool.org.uk/',
 'https://huggingface.co/spaces/Zoro-chi/Book-Recommender',
 'https://github.com/Zoro-chi/Book-Recommender',
 'https://comforterapp.netlify.app/',
 'https://github.com/Zoro-chi/Comforter',
 'https://github.com/Zoro-chi/NetworkSecurity',
 'https://agents-aeqt.onrender.com/',
 'https://github.com/Zoro-chi/agents/tree/main/3_crew/financial_researcher',
 'https://github.com/Zoro-chi/Dog-Vision',
 'https://app.smartparcel.ng/',
 'https://aws.amazon.com/verification',
 '/blog/what-i-actually-learned-building-with-5-agentic-ai-frameworks',
 '/blog/text-preprocessing-in-nlp-a-practical-guide-to-cleaning-your-data',
 '/blog/face-and-skin-condition-analysis-fair-explainable-production-ready-ai',
 '/blog/openai-gpt4-integration',


## First step: Have GPT-4o-mini figure out which links are relevant

### Use a call to gpt-4o-mini to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [6]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [7]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://segun-jr-agbetuyi.vercel.app/"))


Here is the list of links on the website https://segun-jr-agbetuyi.vercel.app/ -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#home
#about
#experience
#projects
#skills
#services
https://github.com/zoro-chi
https://www.linkedin.com/in/segun-jr-agbetuyi/
https://segun-jr-agbetuyi.vercel.app/
https://admissions.au.edu/
https://www.bedfordschool.org.uk/
https://huggingface.co/spaces/Zoro-chi/Book-Recommender
https://github.com/Zoro-chi/Book-Recommender
https://comforterapp.netlify.app/
https://github.com/Zoro-chi/Comforter
https://github.com/Zoro-chi/NetworkSecurity
https://agents-aeqt.onrender.com/
https://github.com/Zoro-chi/agents/tree/main/3_crew/financial_researcher
https://github.com/Zoro-chi/Dog-Vision
https://app.smartparcel.ng/
https://aws.amazon.com/verification
/blog/what-i-actually-learned-buil

In [12]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [13]:
select_relevant_links("https://segun-jr-agbetuyi.vercel.app/")

{'links': [{'type': 'about page',
   'url': 'https://segun-jr-agbetuyi.vercel.app/#about'},
  {'type': 'experience page',
   'url': 'https://segun-jr-agbetuyi.vercel.app/#experience'},
  {'type': 'projects page',
   'url': 'https://segun-jr-agbetuyi.vercel.app/#projects'},
  {'type': 'skills page',
   'url': 'https://segun-jr-agbetuyi.vercel.app/#skills'},
  {'type': 'services page',
   'url': 'https://segun-jr-agbetuyi.vercel.app/#services'},
  {'type': 'LinkedIn profile',
   'url': 'https://www.linkedin.com/in/segun-jr-agbetuyi/'},
  {'type': 'GitHub profile', 'url': 'https://github.com/zoro-chi'}]}

In [14]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [15]:
select_relevant_links("https://segun-jr-agbetuyi.vercel.app/")

Selecting relevant links for https://segun-jr-agbetuyi.vercel.app/ by calling gpt-4o-mini
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://segun-jr-agbetuyi.vercel.app/#about'},
  {'type': 'projects page',
   'url': 'https://segun-jr-agbetuyi.vercel.app/#projects'},
  {'type': 'services page',
   'url': 'https://segun-jr-agbetuyi.vercel.app/#services'},
  {'type': 'experience page',
   'url': 'https://segun-jr-agbetuyi.vercel.app/#experience'}]}

In [16]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-4o-mini
Found 3 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/huggingface'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'company page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-4o-mini

In [17]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [18]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-4o-mini
Found 6 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
Qwen/Qwen3.5-35B-A3B
Updated
6 days ago
•
885k
•
955
Qwen/Qwen3.5-9B
Updated
3 days ago
•
341k
•
435
unsloth/Qwen3.5-35B-A3B-GGUF
Updated
1 day ago
•
792k
•
513
Qwen/Qwen3.5-0.8B
Updated
3 days ago
•
188k
•
251
Qwen/Qwen3.5-27B
Updated
8 days ago
•
467k
•
582
Browse 2M+ models
Spaces
Running
on
Zero
Featured
340
Omni Video Factory
🏆
340
text to video, image to video, video extend
Running
on
Zero
MCP
1.09k
Wan2.2 14B Preview
🐌
1.09k
generate a video from an image with a text prompt
Running
on
Zero
Featured
1.82k
Qwen Image Multiple An

In [19]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [20]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [21]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-4o-mini
Found 6 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.5-35B-A3B\nUpdated\n6 days ago\n•\n885k\n•\n956\nQwen/Qwen3.5-9B\nUpdated\n4 days ago\n•\n341k\n•\n435\nunsloth/Qwen3.5-35B-A3B-GGUF\nUpdated\n1 day ago\n•\n792k\n•\n513\nQwen/Qwen3.5-0.8B\nUpdated\n3 days ago\n•\n188k\n•\n251\nQwen/Qwen3.5-27B\nUpdated\n8 days ago\n•\n467k\n•\n582\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n340\nOmni Video Factory\n🏆\n340\ntext to v

In [22]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [24]:
create_brochure("Segun Agbetuyi", "https://segun-jr-agbetuyi.vercel.app/")

Selecting relevant links for https://segun-jr-agbetuyi.vercel.app/ by calling gpt-4o-mini
Found 7 relevant links


# Segun Jr Agbetuyi - Web Development Expert

Welcome to the world of Segun Jr Agbetuyi, where innovative web applications and AI-powered solutions come to life. With over 5 years of hands-on experience, we specialize in creating scalable, robust web products tailored to meet our clients' needs.

## About Segun Jr Agbetuyi

As a passionate full-stack developer, Segun combines technical expertise with a creative mindset. His journey encompasses a wide range of skills across frontend and backend development, cloud infrastructure, and AI integrations. Comfortably navigating technologies like JavaScript, TypeScript, React, Node.js, and Python, he holds an AWS Developer Associate certification and excels in establishing efficient CI/CD pipelines.

- **Experience**: 5 years
- **Projects Completed**: 10+
- **Client Satisfaction Rate**: 100%
- **Technologies Mastered**: 15+

## Our Services

We pride ourselves on delivering:

- Full-stack web application development
- AI-powered solutions tailored for businesses
- Cloud infrastructure setup and management
- Implementation of solid DevOps practices

## Company Culture

At Segun Jr Agbetuyi, we foster a culture of collaboration and continuous learning. Emphasizing clean, readable code and practical problem-solving, we work closely with designers and product teams to turn conceptual ideas into tangible solutions. Our approach encourages innovation, embraces experimentation, and supports the growth of fellow developers through mentorship and open-source contributions.

## Join Us

We are always on the lookout for talented developers and tech enthusiasts who are eager to thrive in a remote, collaborative environment. If you're interested in joining a team that values curiosity and creativity, reach out to us!

## Connect with Us

**Email**: jagbetuyi001@gmail.com  
**Location**: Remote  
**Follow**: [GitHub](#) | [LinkedIn](#)

Let's build something great together!

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [25]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [27]:
stream_brochure("Akatsuki", "https://segun-jr-agbetuyi.vercel.app/")

Selecting relevant links for https://segun-jr-agbetuyi.vercel.app/ by calling gpt-4o-mini
Found 8 relevant links


# Akatsuki Company Brochure

## About Us
Welcome to Akatsuki! We are a cutting-edge technology firm driven by innovation and excellence in the development of web applications and AI-powered solutions. Founded by Segun Jr Agbetuyi, a passionate full-stack developer with over 5 years of experience, our mission is to build scalable digital products that enhance user experience and streamline operations for businesses.

## Our Expertise
At Akatsuki, we specialize in full-stack development, utilizing modern technologies such as:
- **Frontend**: React, Next.js
- **Backend**: Node.js, Python
- **Cloud Technologies**: AWS (certified AWS Developer Associate)
- **CI/CD Pipelines**: Enhancing deployment processes
- **AI & ML Integration**: Crafting intelligent solutions

With over 10 successful projects completed and a robust approach to client satisfaction, we ensure delivery that meets our clients' needs.

## Company Culture
Our culture is built on collaboration, innovation, and a commitment to continuous learning. We foster a work environment where ideas are freely exchanged, and creativity thrives. Our team is passionate about writing clean, readable code and keeps the development process simple and efficient. We proudly support side projects and open-source contributions, encouraging our team to explore new ideas and technologies.

## Career Opportunities
Akatsuki is always on the lookout for talented individuals who share our passion for technology and innovation. We offer:
- A remote-first working environment
- Opportunities for continuous professional development
- A culture that encourages collaboration and personal growth
- The chance to work on diverse projects with cutting-edge technologies

Join us in our journey to build impactful solutions and help us shape the future of technology!

## Connect with Us
We’re excited to hear from you whether you are a prospective customer, investor, or recruit! 

- 📧 Email: [jagbetuyi001@gmail.com](mailto:jagbetuyi001@gmail.com)
- 🌐 Visit our website and explore our portfolio.

**Together, let’s build something incredible!**

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>